In [221]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import json
import copy
from scipy.stats import wilcoxon, mannwhitneyu
import pickle as pkl

import nibabel as nib

import matplotlib.pyplot as plt 

In [245]:
def preprocess_mask_labels(mask):
    # whole tumour
    mask_WT = mask.copy()
    mask_WT[mask_WT == 1] = 1
    mask_WT[mask_WT == 2] = 1
    mask_WT[mask_WT == 3] = 1
    # include all tumours 

    # NCR / NET - LABEL 1
    mask_TC = mask.copy()
    mask_TC[mask_TC == 1] = 1
    mask_TC[mask_TC == 2] = 0
    mask_TC[mask_TC == 3] = 1
    # exclude 2 / 4 labelled tumour 

    # ET - LABEL 4 
    mask_ET = mask.copy()
    mask_ET[mask_ET == 1] = 0
    mask_ET[mask_ET == 2] = 0
    mask_ET[mask_ET == 3] = 1
    # exclude 2 / 1 labelled tumour 

    mask = np.stack([mask_WT, mask_TC, mask_ET])
    
    return mask 

def get_tumor_slices(mri_mask):
    """
    Extracts slices from a 3D MRI mask where the tumor exists.

    Parameters:
    mri_mask (numpy.ndarray): A 3D NumPy array representing the MRI mask.

    Returns:
    list of numpy.ndarray: A list of 2D slices containing the tumor.
    """
    tumor_slices = []

    # Iterate through each slice
    for i in range(mri_mask.shape[2]):
        _slice = mri_mask[:, :, i]
        
        # Check if the slice contains tumor (non-zero values)
        if np.any(_slice):
            tumor_slices.append(i)

    return tumor_slices

def read_MRI(dataset, patient_id):
    if dataset == 'Brats2020':
        baseloc = '../../input/BraTS2020_TrainingData/MICCAI_BraTS2020_TrainingData/'
        pefix = 'BraTS20_Training_' + patient_id + '/' + 'BraTS20_Training_' + patient_id
        suffixs = ['_flair.nii','_t2.nii', '_t1.nii', '_t1ce.nii', '_seg.nii']
    elif dataset == 'Brats2023':
        baseloc = '../../input/Brats2023/ASNR-MICCAI-BraTS2023-GLI-Challenge-TrainingData/'
        pefix = 'BraTS-GLI-' + patient_id + '/' + 'BraTS-GLI-' + patient_id
        result_loc = '../../Results/Result/Vanilla_Unet/BraTS-GLI-' + patient_id
        suffixs = ['-t2f.nii.gz','-t2w.nii.gz', '-t1n.nii.gz', '-t1c.nii.gz', '-seg.nii.gz', '-seg..npz']

    flair_filename = baseloc + pefix + suffixs[0]
    flair_img_f = nib.load(flair_filename)
    flair_img = np.asarray(flair_img_f.dataobj)

    t2_filename = baseloc + pefix + suffixs[1]
    t2_img_f = nib.load(t2_filename)
    t2_img = np.asarray(t2_img_f.dataobj)

    t1_filename = baseloc + pefix + suffixs[2]
    t1_img_f = nib.load(t1_filename)
    t1_img = np.asarray(t1_img_f.dataobj)

    t1ce_filename = baseloc + pefix + suffixs[3]
    t1ce_img_f = nib.load(t1ce_filename)
    t1ce_img = np.asarray(t1ce_img_f.dataobj)
    
    mask_filename = baseloc + pefix + suffixs[4]
    mask_img_f = nib.load(mask_filename)
    mask_img = np.asarray(mask_img_f.dataobj)
    
    output_filename = result_loc + suffixs[4]
    output_img_f = nib.load(output_filename)
    output_img = np.asarray(output_img_f.dataobj)
    
    prob_filename = result_loc + suffixs[5]
    probablity_img = np.load(prob_filename, allow_pickle=True)
    probablity_img = probablity_img['arr_0'][0]
    probablity_img = np.moveaxis(probablity_img, (0, 1, 2, 3), (0, 3, 2, 1))
    
    return flair_img, t2_img, t1_img, t1ce_img, mask_img, output_img, probablity_img

def calculate_probability(dataset, patient_id, radius):
    flair_img, t2_img, t1_img, t1ce_img, mask_img, output_img, probablity_img = read_MRI(dataset, patient_id)
    
    masks = preprocess_mask_labels(mask_img)
#     mask_WT, mask_TC, mask_ET = mask[0], mask[1], mask[2]
    
    struct = np.ones((2*radius+1, 2*radius+1, 2*radius+1))
    
    all_probabilities = []

    for i in range(len(masks)):
        mask = masks[i]
        
        prob = 1 / (1 + np.exp(-probablity_img[i]))
        
        dilated_mask = binary_dilation(mask,struct)
        eroded_mask = binary_erosion(mask,struct)
        boundary_region = dilated_mask ^ eroded_mask
        
        dilated_probabilities = prob[dilated_mask]
        eroded_probabilities = prob[eroded_mask]
        boundary_probabilities = prob[boundary_region]
        
        all_probabilities.append(dilated_probabilities)
        all_probabilities.append(eroded_probabilities)
        all_probabilities.append(boundary_probabilities)
        
    return all_probabilities
        
    
    
def load_unet_result(path, _print):
    unet_df = pd.read_csv(path, index_col = 'Unnamed: 0')
    index_values = []
    for _index in unet_df.index:
        index_values.append(_index.split('-seg')[0])

    unet_df.index = index_values  
    unet_df.drop(['WT jaccard', 'TC jaccard', 'ET jaccard'], axis = 1, inplace = True)
    summary_unet_df = pd.DataFrame(zip(unet_df.mean().values.tolist(), 
                                       unet_df.std().values.tolist()), 
                                   columns = ['mean', 'std'], index = unet_df.columns)
    if _print:
        print("****UNet******")
        print(summary_unet_df)
    return summary_unet_df, unet_df

# Load the JSON file
def load_nnunet_result(path, _print):
    with open(path, 'r') as file:
        data = json.load(file)

    WT = []
    TC = []
    ET = []
    file_name = []
    for case in data['metric_per_case']:
        WT.append(case['metrics']['(2, 1, 3)']['Dice'])
        TC.append(case['metrics']['(2, 3)']['Dice'])
        ET.append(case['metrics']['(3,)']['Dice'])
        file_name.append(case['reference_file'].split('/')[-1].split('.')[0])

    nnunet_df = pd.DataFrame(zip(WT, TC, ET), columns = ['WT dice', 'TC dice', 'ET dice'], 
                             index = file_name)
    summary_nnunet_df = pd.DataFrame(zip(nnunet_df.mean().values.tolist(), 
                                       nnunet_df.std().values.tolist()), 
                                   columns = ['mean', 'std'], index = nnunet_df.columns)
    if _print:
        print("****nnUNet******")
        print(summary_nnunet_df)
    return summary_nnunet_df, nnunet_df

def load_TransBTS_result(path, _print):
    with open(path, 'r') as file:
        data = json.load(file)

    WT = []
    TC = []
    ET = []
    file_name = []
    for case_id in data.keys():
        case = data[case_id]
        WT.append(case['WT'][0])
        TC.append(case['TC'][0])
        ET.append(case['ET'][0])
        file_name.append(case_id)

    TransBTS_df = pd.DataFrame(zip(WT, TC, ET), columns = ['WT dice', 'TC dice', 'ET dice'], 
                             index = file_name)
    summary_TransBTS_df = pd.DataFrame(zip(TransBTS_df.mean().values.tolist(), 
                                       TransBTS_df.std().values.tolist()), 
                                   columns = ['mean', 'std'], index = TransBTS_df.columns)
    if _print:
        print("****TransBTS******")
        print(summary_TransBTS_df)
    return summary_TransBTS_df, TransBTS_df

def read_results(_print=True):
    path = '../../Results/Result/Vanilla_Unet/Unet_test_dice.csv'
    summary_unet_df, unet_df = load_unet_result(path, _print)

    path = '../../Results/Result/nnUnet/nnUNetTrainer/summary.json'
    summary_da_nnunet_df, nnunet_da_df = load_nnunet_result(path, _print)

    path = '../../Results/Result/nnUnet/nnUNetTrainerNoDA/summary.json'
    summary_noda_nnunet_df, nnunet_noda_df = load_nnunet_result(path, _print)

    path = '../../Results/Result/TransBTS/submission/TransBTS2023-11-03/TransBTS_summary.json'
    summary_TransBTS_df, TransBTS_df = load_TransBTS_result(path, _print)
    return summary_unet_df, unet_df, summary_noda_nnunet_df, nnunet_noda_df, summary_da_nnunet_df, nnunet_da_df, summary_TransBTS_df, TransBTS_df

def get_overlaps(unet_df, TransBTS_df, nnunet_noda_df, dice_threshold, dice_score):
    unet_df_sub = unet_df[unet_df[dice_score] < dice_threshold]
    TransBTS_df_sub = TransBTS_df[TransBTS_df[dice_score] < dice_threshold]
    nnunet_noda_df_sub = nnunet_noda_df[nnunet_noda_df[dice_score] < dice_threshold]

    unet_df_sub_subjects = unet_df_sub.index.values.tolist()
    TransBTS_df_sub_subjects = TransBTS_df_sub.index.values.tolist()
    nnunet_noda_df_sub_subjects = nnunet_noda_df_sub.index.values.tolist()

    all_overlaps = list(set(unet_df_sub_subjects) & set(TransBTS_df_sub_subjects) & set(nnunet_noda_df_sub_subjects))
#     print('all overlap', len(all_overlaps))

    unet_nnunet_overlaps = list(set(unet_df_sub_subjects) & set(nnunet_noda_df_sub_subjects))
#     print('unet-nnunet overlap', len(unet_nnunet_overlaps))

    unet_TransBTS_overlaps = list(set(unet_df_sub_subjects) & set(TransBTS_df_sub_subjects))
#     print('unet-TransBTS overlap', len(unet_TransBTS_overlaps))

    nnunet_TransBTS_overlaps = list(set(TransBTS_df_sub_subjects) & set(nnunet_noda_df_sub_subjects))
#     print('nnunet-TransBTS overlap', len(nnunet_TransBTS_overlaps))
    
    return all_overlaps, unet_nnunet_overlaps, unet_TransBTS_overlaps, nnunet_TransBTS_overlaps


def cliffs_delta(x, y):
    n_x = len(x)
    n_y = len(y)
    N_gr = sum(xi > yi for xi in x for yi in y)
    N_ls = sum(xi < yi for xi in x for yi in y)
    return (N_gr - N_ls) / (n_x * n_y)

In [246]:
dataset = 'Brats2023'
radius = 1
data_path = '../../input/BraTS2023/ASNR-MICCAI-BraTS2023-GLI-Challenge-TrainingData/'
result_path = '../../Results/Analysis_Results/probability/GLI-Image_intensity_tumor_vs_non_tumor.pkl'
patient_ids = [f for f in listdir(data_path) if not isfile(join(data_path, f))]

probs = []
all_patient_ids = []

# for _id in patient_ids:
#     print(_id)
#     all_patient_ids.append(_id)
#     patient_id = _id.split('GLI-')[1]
#     probs = calculate_probability(dataset, patient_id, radius)
#     probabilitis.append(probs)
    
# probabilitis_df = pd.DataFrame(probabilitis, 
#                                columns = ['WT_dilated', 'WT_eroded', 'WT_boundary', 
#                                           'TC_dilated', 'TC_eroded', 'TC_boundary', 
#                                           'ET_dilated', 'ET_eroded', 'ET_boundary'],
#                                index = all_patient_ids)
    
# probabilitis_df.to_csv(result_path)

In [247]:
probabilitis_df = pd.read_pickle(result_path)

In [254]:
probabilitis_median = []
ids = [] 
for index in probabilitis_df.index:
    ids.append(index)
    WT_dilated_median = np.mean(probabilitis_df.loc[index, 'WT_dilated'])
    WT_eroded_median = np.mean(probabilitis_df.loc[index, 'WT_eroded'])
    WT_boundary_median = np.mean(probabilitis_df.loc[index, 'WT_boundary'])
    TC_dilated_median = np.mean(probabilitis_df.loc[index, 'TC_dilated'])
    TC_eroded_median = np.mean(probabilitis_df.loc[index, 'TC_eroded'])
    TC_boundary_median = np.mean(probabilitis_df.loc[index, 'TC_boundary'])
    ET_dilated_median = np.mean(probabilitis_df.loc[index, 'ET_dilated'])
    ET_eroded_median = np.mean(probabilitis_df.loc[index, 'ET_eroded'])
    ET_boundary_median = np.mean(probabilitis_df.loc[index, 'ET_boundary'])
    probabilitis_median.append([WT_dilated_median, WT_eroded_median, WT_boundary_median, 
                                TC_dilated_median, TC_eroded_median, TC_boundary_median, 
                                ET_dilated_median, ET_eroded_median, ET_boundary_median])

probabilitis_median_df = pd.DataFrame(probabilitis_median, 
                                  columns=['WT_dilated', 'WT_eroded', 'WT_boundary', 
                                          'TC_dilated', 'TC_eroded', 'TC_boundary', 
                                          'ET_dilated', 'ET_eroded', 'ET_boundary'],
                                  index = ids)

probabilitis_df.to_pickle('/proj/arise/arise/suvodeep/Result/GLI-Image_intensity_tumor_vs_non_tumor.pkl')